# 🧠 Handwritten Note Digitizer using Gemini AI 📄✍️
### Capstone Project for Google GenAI Intensive 2025
---

### 👥 Team Members:
- **Bhanu Prasad Thota**  
  📧 [bhanuprasadt27@gmail.com](mailto:bhanuprasadt27@gmail.com)

- **Parvash Choudhary Talluri**  
  📧 [talluriparvashchoudhary2001@gmail.com](mailto:talluriparvashchoudhary2001@gmail.com)


## 🔍 Project Overview

In this project, we address a common problem: handwritten notes are often unstructured, hard to search, and difficult to revisit after some time. Our goal is to build an end-to-end pipeline that takes scanned handwritten notes as input and generates a polished, readable, enriched, and exportable LaTeX document — using Google's Gemini models.

This project is part of the Google GenAI Intensive Capstone, and demonstrates several core capabilities of Generative AI:

- 🖼️ Image Understanding (Handwritten Text Recognition)
- 🧠 Document Understanding (Cohesion, Enrichment)
- ✍️ Structured Output Generation (LaTeX Formatting)


## 🧠 Gen AI Capabilities Demonstrated

✅ **Image Understanding**: Interpreting scanned handwritten images using Gemini.

✅ **Document Understanding**: Correcting grammar, adding missing context, and restructuring fragmented content.

✅ **Structured Output**: Generating LaTeX code for final formatted output.

✅ **Long Context Processing**: Gemini handles multiple pages at once to maintain overall cohesion.

✅ **(Optional) Image Prompting**: Gemini-generated image prompts used to create educational diagrams (uploaded separately).


## 📂 Input: Scanned Note Pages

We use a folder of handwritten notes (`/kaggle/input/pagesimgs/`) which contain scanned pages of lecture-style content. These were uploaded manually through Kaggle's file interface.

Each image is processed using Gemini to extract and enrich the content.


## 🛠️ Pipeline Steps

1. 📤 **Upload handwritten note images**
2. 🧠 **Use Gemini Pro (1.5)** to extract and enhance content
3. 🧹 **Retry logic** to handle Gemini's API rate limits
4. 📄 **Merge and reorder** all extracted text
5. 🧾 **Ask Gemini to generate valid LaTeX**
6. 📥 **Compile LaTeX to PDF**


# 🔧 Step 1: Import Required Libraries

This notebook uses several key libraries to process handwritten notes, communicate with Google's Gemini API, and render final outputs:

In [ ]:
# 📦 Core Python Libraries
import os
import time

# 🖼️ Image Handling
from PIL import Image
import matplotlib.pyplot as plt

# 📋 Data Handling (optional, useful for debugging or table outputs)
import pandas as pd

# 🧠 Google Generative AI SDK
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# 📄 PDF Generation
from IPython.display import Markdown, display, FileLink

import time
from PIL import Image

In [ ]:
# Checkpoint: save extracted text after each page to avoid losing progress on rate limit errors
import json
CHECKPOINT_FILE = "/kaggle/working/checkpoint.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}

def save_checkpoint(data):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(data, f, indent=2)

checkpoint = load_checkpoint()
print(f"Checkpoint loaded: {len(checkpoint)} pages already processed")

# 🖼️ Step 2: Verify Uploaded Handwritten Note Images

Before processing, we first inspect the uploaded scanned images of handwritten notes.  
All images should be located inside the `/kaggle/input/pagesimgs` directory.

In [ ]:
# 📂 Define image directory
image_dir = "/kaggle/input/pagesimgs"

# 🖼️ List all image files in the directory
print("📁 Uploaded handwritten note images:\n")
import re
def natural_sort_key(f): return [int(c) if c.isdigit() else c.lower() for c in re.split(r"(\d+)", f)]
image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))], key=natural_sort_key)

for file in image_files:
    print(f"– {file}")

print(f"\n✅ Total images found: {len(image_files)}")


# ✅ Step 3: Authenticate and Validate Gemini API Key

We now load the API key from **Kaggle Secrets** and initialize the Gemini model.  
A simple prompt is run to confirm the connection.

In [ ]:
# 🔐 Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GOOGLE_API_KEY")

# ✅ Check if the key exists
if api_key and len(api_key) > 10:
    print("✅ Gemini API key loaded successfully.")
else:
    raise ValueError("❌ API key is missing or invalid. Please check your Kaggle secret setup.")

# ⚙️ Configure Gemini client
genai.configure(api_key=api_key)

# 🧠 Initialize Gemini model
try:
    model = genai.GenerativeModel("gemini-2.0-flash")
    print("🧠 Gemini model initialized.")
    
    # ✅ Optional: Run a safe test query
    try:
        test_response = model.generate_content("Hello Gemini, just confirming you're ready.")
        print("✅ Gemini is responding: ", test_response.text.strip()[:60], "...")
    except Exception as test_err:
        print("⚠️ Gemini model initialized, but test prompt failed:")
        print(test_err)

except Exception as e:
    raise RuntimeError(f"❌ Gemini model initialization failed: {e}")

# 🖼️ Step 4: Preview Each Handwritten Image

Below, we visually inspect the uploaded images to ensure they're readable and correctly ordered.  

In [ ]:
# 🖼️ Display each uploaded handwritten image

# Get all image files
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)
    img = Image.open(img_path)

    # Resize to avoid rendering large images in Kaggle
    img.thumbnail((600, 600))

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"📝 {img_file}")
    plt.show()


# 🧩 Utility: Rate-Limit Safe Gemini Generation

This helper function wraps calls to `model.generate_content()` and:

- Detects `429` rate-limit errors
- Waits and retries up to 5 times
- Ensures smooth uninterrupted Gemini usage


In [ ]:
# 🔁 Helper: Retry-safe Gemini generation function
def safe_generate(prompt, max_retries=5, wait_time=45):
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            if "429" in str(e):
                print(f"⏳ Rate limit hit. Retrying in {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ Error: {e}")
                return ""
    print("❌ Max retries exceeded.")
    return ""

# 🧠 Step 5: Extract Text from Handwritten Notes using Gemini

In this step, we send each scanned image to **Gemini 1.5 Pro**, asking it to:

- Understand and read handwritten content
- Correct grammar and spelling
- Return a clean, readable version of each page's notes

In [ ]:
# Store extracted clean texts in an array
extracted_texts = []

# Process all images
for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)
    img = Image.open(img_path)

    prompt = [
        "This is a handwritten page of notes. Please extract the text and rewrite it with grammar/spelling corrections.",
        img
    ]

    print(f"🔍 Processing {img_file}...")
    clean_text = safe_generate(prompt)
    if clean_text:
        print(f"✅ Extracted {len(clean_text)} characters from {img_file}\n")
    else:
        print(f"⚠️ No text extracted from {img_file}")
    
    extracted_texts.append(clean_text)


# 🧠 Step 6: Detect Hand-Drawn Diagrams and Sketches from Scanned Notes

In this step, we use **Gemini Vision API** to analyze each scanned note image and **identify any hand-drawn diagrams, sketches, or flowcharts**.

Our primary goals were to:
- Detect important **visual elements** like conceptual diagrams, vector space plots, or architectural flows
- **Ignore text content**, focusing only on visuals
- Use Gemini's understanding to either:
  - 🖼️ **Crop out** the relevant diagram from the scanned image (for reuse in our output PDF), or
  - 🎨 **Regenerate** a clean version of the diagram using an AI image generation model based on Gemini's description

### ❗ Limitation
Due to current API limitations:
- Gemini does not return image bounding boxes or allow image cropping based on vision understanding
- The Kaggle environment does not support Gemini's image generation model directly (e.g., `gemini-2.0-flash-exp-image-generation`)
- Rate-limits on the free API prevented extensive prompt chaining or regeneration trials

Despite this, we demonstrate Gemini's **semantic diagram recognition capability**, and lay the foundation to:
- Extract and label diagrams for downstream use
- Potentially generate visual content in local or cloud environments (e.g., Colab + Gemini Studio or Vertex AI)

This sets up an exciting future enhancement for fully automated lecture/meeting note digitization with both text and visuals 🎯


In [ ]:
# Array to store Gemini responses
diagram_descriptions = []

# Loop through each image file
for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)
    img = Image.open(img_path)

    # Prompt focused only on diagrams
    prompt = [
        f"This is a scanned page named {img_file}. Please ignore the text.",
        "Focus only on any hand-drawn diagrams, sketches, or illustrations.",
        "Describe the diagram clearly: what elements are present? What is it showing?",
        img
    ]

    print(f"🔍 Analyzing diagrams in: {img_file}...")
    diagram_response = safe_generate(prompt)

    diagram_descriptions.append({
        "filename": img_file,
        "description": diagram_response
    })

    print("🧠 Gemini Diagram Interpretation:")
    print(diagram_response)
    print("-" * 80)


# 🧠 Step 7: Reorder, Enrich, and Structure Extracted Notes using Gemini

After extracting raw text content from the scanned handwritten pages, we now use **Gemini Pro** to:

- 🧩 **Reorder** the content if the pages are out of sequence  
- 🔗 **Fix fragmented transitions** that may have occurred due to disjointed note-taking  
- 💬 **Enrich the notes** by adding minimal but relevant context for clarity  
- 🧠 **Preserve technical structure** such as:
  - Code blocks
  - Formulas
  - Bullet points
  - Headings

In [ ]:
# Join all page contents together
joined_notes = "\n\n".join([
    f"Page {i+1}:\n{text}" +
    (f"\n\n[Diagram: {diagram_descriptions[i]["description"]}]" if i < len(diagram_descriptions) and diagram_descriptions[i]["description"] and "No diagrams" not in diagram_descriptions[i]["description"] else "")
    for i, text in enumerate(extracted_texts) if text.strip() != ""
])

# Prompt Gemini to clean, reorder, and enrich
final_prompt = f"""
You are an expert in academic note-taking and organization.

Below are scanned handwritten notes extracted and cleaned using OCR. The content might be out of order, redundant, or missing context. Please:

1. Reorder the pages logically (if needed).
2. Fix transitions to improve cohesion.
3. Add minimal but relevant missing information for smooth understanding.
4. Output the entire content as a clean, structured, multi-section document (NO page labels like Page 1/Page 2).
5. Preserve technical accuracy and structure (e.g., code blocks, formulas, bullet points, headings).

Notes:
{joined_notes}
"""

# Send to Gemini for enrichment
response = model.generate_content(final_prompt)

# Display the final cleaned & enriched version
from IPython.display import Markdown
Markdown(response.text)


# 📄 Step 8: Generate Markdown-Formatted Output for PDF Export

In this step, we instruct **Gemini Pro** to return the cleaned and enriched notes in **strict Markdown format** — optimized for direct conversion to PDF.

This ensures that:
- The final document is semantically structured
- We can easily convert it to PDF using `WeasyPrint` in the next step
- The formatting (headings, bullets, math blocks) is preserved in the final output


In [ ]:
final_promptpdf = f"""
You are an expert in technical writing and Markdown formatting.

Below are scanned and cleaned handwritten notes. Please:
1. Reorder the content logically if needed.
2. Fix transitions to improve cohesion and clarity.
3. Format the output as clean, structured **Markdown** for PDF export:
   - Use # and ## for section headings
   - Use bullet points where applicable
   - Wrap formulas in \\[ ... \\] LaTeX math blocks
   - Include proper indentation, spacing, and structure

Output only Markdown — no explanations or extra commentary.

Notes:
{joined_notes}
"""
markdown_response = model.generate_content(final_promptpdf)


# 🖨️ Step 9: Convert Markdown Output to PDF

Now that we’ve received the cleaned and well-formatted notes in Markdown from Gemini, we generate a **final downloadable PDF** using the following steps:

1. 📄 **Save** the Markdown-formatted output (`response.text`) to a `.md` file.
2. 🔁 **Convert** the Markdown into HTML using `markdown2`.
3. 📦 **Render** the HTML to a PDF using `WeasyPrint`, which supports full styling inside the Kaggle environment.
4. 🔗 **Generate a download link** for the PDF directly inside the notebook.

In [ ]:
# 📦 Install required libraries (only needed once)
!pip install -q markdown2 weasyprint


# Required libraries
from IPython.display import FileLink
import markdown2
from weasyprint import HTML

# Use Gemini's markdown-formatted output
enriched_notes = markdown_response.text

# Save as .md file
with open("final_notes.md", "w") as f:
    f.write(enriched_notes)

# Convert Markdown → HTML → PDF
html_content = markdown2.markdown(enriched_notes, extras=["fenced-code-blocks"])
HTML(string=html_content).write_pdf("final_notes.pdf")

# Show download link
FileLink("final_notes.pdf")



### ⚠️ Note: PDF Formatting Limitations

While the final PDF was successfully generated using the Markdown + WeasyPrint workflow, it’s important to acknowledge a few **limitations**:

- The **visual formatting** of the document (especially math blocks, spacing, and code structure) is not always preserved as expected
- **Mathematical equations** wrapped in `\[` `\]` are displayed as plain text and not rendered like LaTeX
- Complex structures such as aligned equations, tables, and inline math are not visually accurate

---

### 🧠 Why LaTeX Would Have Been Better

Given the technical nature of the notes (NLP models, formulas, token matrices, etc.), this project would have significantly benefited from **LaTeX formatting**, which provides:

- Beautifully rendered math using `\(\)` or `\[ \]`
- Structured document formatting
- Better control over spacing, page breaks, and table layout

---

### ❗ Kaggle Limitation

Unfortunately, **Kaggle currently does not support `pdflatex` or full LaTeX compilation**, which forced us to use Markdown + HTML → PDF as a workaround.

> 💡 In a production or publication-grade setup, we would highly recommend converting the cleaned notes into LaTeX and compiling with Overleaf or a local TeX engine for maximum quality.

This workaround still allowed us to complete the pipeline end-to-end using only Kaggle and Gemini Pro, demonstrating the feasibility of automated note digitization.


### 📎 Additional PDF Attached (LaTeX Rendered)

Due to **Kaggle's limitations** in supporting LaTeX compilation (e.g., `pdflatex` is not available), the main pipeline in this notebook uses **Markdown + WeasyPrint** for generating the final PDF.

While this approach works, it has several formatting limitations:
- Mathematical expressions are **not visually rendered** (only shown as raw LaTeX)
- Advanced formatting like **equation alignment**, **tables**, and **fine-grained spacing** is not supported
- The overall document structure lacks **typographic polish**


📥 **Attached below is the LaTeX-based PDF for reference**, showing what the final output *should* ideally look like with full formatting capabilities enabled.

👉 [Click here to view/download the PDF (Google Drive)](https://drive.google.com/file/d/1nfdA5SLA91GlgNgj6ujyZgzOvhQEn2xW/view)